# ds004752 V5.6 Tranche 2 Scaffold Artifacts

Notebook nay chay tu tren xuong duoi de sinh va review bon artifact scaffold cua V5.6 Tranche 2:

- `v56_split_registry`
- `v56_feature_provenance`
- `v56_control_registry`
- `v56_leaderboard`

Ranh gioi khoa hoc: notebook nay khong train model, khong chay comparator, khong tinh efficacy metric, va khong mo claim.


## 1. Mount Google Drive

Artifacts va Gate 0 run duoc doc/ghi duoi `/content/drive/MyDrive/eeg-ds004752`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone hoac update repo vao `/content`

Notebook chay code tu `/content/eeg-ds004752` de tranh dung nham ban cu trong Drive.


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', 'main'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR)


## 3. Xac nhan runtime co Tranche 2 scaffold code

Cell nay bat loi neu Colab van dung commit cu chua co `v56-scaffold` hoac script Tranche 2.


In [ ]:
%cd /content/eeg-ds004752
!git log --oneline -8

from pathlib import Path

required_files = [
    Path('src/v56/runner.py'),
    Path('src/v56/artifacts.py'),
    Path('tests/unit/test_v56_cli.py'),
    Path('bootstrap/run_v56_tranche2_scaffold.sh'),
    Path('docs/22_v56_tranche2_scaffold_runbook_2026-04-24.md'),
]
missing = [str(p) for p in required_files if not p.exists()]
print({'missing_required_files': missing})
assert not missing, missing

cli_text = Path('src/cli.py').read_text(encoding='utf-8')
assert 'v56-scaffold' in cli_text
assert 'run_v56_scaffold' in cli_text
print('V5.6 Tranche 2 scaffold code is present.')


## 4. Cau hinh source-of-truth va output roots

Gate 0 run phai la full-cohort signal-ready run da review: `20260424T100159866284Z`.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260424T100159866284Z'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts'

print('GATE0_RUN =', GATE0_RUN)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('gate0_exists =', GATE0_RUN.exists())
assert GATE0_RUN.exists(), f'Missing Gate 0 run: {GATE0_RUN}'


## 5. Preflight Gate 0 authoritative artifacts

Doc `manifest.json`, `cohort_lock.json`, va `materialization_report.json` truoc khi tao Tranche 2 artifacts.


In [ ]:
import json

manifest = json.loads((GATE0_RUN / 'manifest.json').read_text())
cohort_lock = json.loads((GATE0_RUN / 'cohort_lock.json').read_text())
materialization = json.loads((GATE0_RUN / 'materialization_report.json').read_text())

gate0_summary = {
    'manifest_status': manifest.get('manifest_status'),
    'primary_eligibility_status': manifest.get('primary_eligibility_status'),
    'gate0_blockers': manifest.get('gate0_blockers'),
    'signal_status': manifest.get('signal_audit', {}).get('status'),
    'sessions_checked': manifest.get('signal_audit', {}).get('sessions_checked'),
    'mat_files_checked': manifest.get('signal_audit', {}).get('mat_files_checked'),
    'cohort_lock_status': cohort_lock.get('cohort_lock_status'),
    'n_primary_eligible': cohort_lock.get('n_primary_eligible'),
    'edf_materialized': materialization['payloads']['edf']['materialized_count'],
    'edf_total': materialization['payloads']['edf']['count'],
    'mat_materialized': materialization['payloads']['mat']['materialized_count'],
    'mat_total': materialization['payloads']['mat']['count'],
}
print(json.dumps(gate0_summary, indent=2))

assert gate0_summary['manifest_status'] == 'signal_audit_ready', gate0_summary
assert gate0_summary['cohort_lock_status'] == 'signal_audit_ready', gate0_summary
assert gate0_summary['gate0_blockers'] == [], gate0_summary
assert gate0_summary['signal_status'] == 'ok', gate0_summary
assert gate0_summary['sessions_checked'] == 68, gate0_summary
assert gate0_summary['mat_files_checked'] == 15, gate0_summary
assert gate0_summary['n_primary_eligible'] == 15, gate0_summary
assert gate0_summary['edf_materialized'] == gate0_summary['edf_total'], gate0_summary
assert gate0_summary['mat_materialized'] == gate0_summary['mat_total'], gate0_summary
print('Gate 0 authoritative run is signal-ready.')


## 6. Chay V5.6 Tranche 2 scaffold artifact generation

Cell nay chi sinh scaffold artifacts. Khong model training, khong comparator execution, khong efficacy metrics.


In [ ]:
%cd /content/eeg-ds004752
!bash bootstrap/run_v56_tranche2_scaffold.sh {GATE0_RUN} {OUTPUT_ROOT}


## 7. Lay latest run dirs cua 4 artifact families

Cell nay doc `latest.txt` cua moi artifact family vua tao.


In [ ]:
artifact_roots = {
    'v56_split_registry': OUTPUT_ROOT / 'v56_split_registry',
    'v56_feature_provenance': OUTPUT_ROOT / 'v56_feature_provenance',
    'v56_control_registry': OUTPUT_ROOT / 'v56_control_registry',
    'v56_leaderboard': OUTPUT_ROOT / 'v56_leaderboard',
}

artifact_dirs = {}
for family, root in artifact_roots.items():
    latest = root / 'latest.txt'
    assert latest.exists(), f'Missing latest pointer for {family}: {latest}'
    run_dir = Path(latest.read_text().strip())
    assert run_dir.exists(), f'Missing run dir for {family}: {run_dir}'
    artifact_dirs[family] = run_dir

for family, run_dir in artifact_dirs.items():
    print(f'{family}: {run_dir}')


## 8. Doc artifact JSON va summaries

Cell nay load artifact body + summary cua tung family de review va assert.


In [ ]:
artifacts = {}
summaries = {}
for family, run_dir in artifact_dirs.items():
    artifact_path = run_dir / f'{family}.json'
    summary_path = run_dir / f'{family}_summary.json'
    report_path = run_dir / f'{family}_report.md'
    scaffold_record_path = run_dir / 'v56_benchmark_scaffold_record.json'
    for path in [artifact_path, summary_path, report_path, scaffold_record_path]:
        assert path.exists(), f'Missing expected file: {path}'
    artifacts[family] = json.loads(artifact_path.read_text())
    summaries[family] = json.loads(summary_path.read_text())

compact = {
    family: {
        'status': artifacts[family].get('status'),
        'claim_closed': artifacts[family].get('claim_closed'),
        'gate0_manifest_status': artifacts[family].get('gate0_manifest_status'),
        'cohort_lock_status': artifacts[family].get('cohort_lock_status'),
        'n_primary_eligible': artifacts[family].get('n_primary_eligible'),
        'summary_status': summaries[family].get('status'),
        'summary_claim_closed': summaries[family].get('claim_closed'),
    }
    for family in artifacts
}
print(json.dumps(compact, indent=2))


## 9. Integrity assertions

Neu cell nay pass, scaffold artifacts dung ranh gioi claim-closed va chua chay model/metrics.


In [ ]:
expected_status = {
    'v56_split_registry': 'pending_registry_lock',
    'v56_feature_provenance': 'pending_feature_provenance_population',
    'v56_control_registry': 'pending_control_execution',
    'v56_leaderboard': 'pending_comparator_execution',
}

for family, artifact in artifacts.items():
    assert artifact.get('status') == expected_status[family], (family, artifact.get('status'))
    assert artifact.get('gate0_manifest_status') == 'signal_audit_ready', family
    assert artifact.get('cohort_lock_status') == 'signal_audit_ready', family
    assert artifact.get('n_primary_eligible') == 15, family
    assert summaries[family].get('claim_closed') is True, family

split = artifacts['v56_split_registry']
tracks = {row['id']: row for row in split['tracks']}
assert split['test_time_inference'] == 'scalp_eeg_only', split
assert tracks['scalp_only_primary']['privileged_train_time'] is False, tracks
assert tracks['privileged_train_scalp_test']['privileged_train_time'] is True, tracks

controls = artifacts['v56_control_registry']
blocking_tiers = [tier['id'] for tier in controls['tiers'] if tier['claim_blocking']]
assert blocking_tiers == ['data_integrity', 'control_adequacy', 'reporting'], blocking_tiers

leaderboard = artifacts['v56_leaderboard']
assert leaderboard['primary_target_id'] == 'A4_privileged', leaderboard
assert all(row['run_status'] == 'pending_not_run' for row in leaderboard['rows']), leaderboard['rows']
assert not any('balanced_accuracy' in row for row in leaderboard['rows']), leaderboard['rows']

print('V5.6 Tranche 2 scaffold artifacts passed integrity assertions.')


## 10. In report ngan cho tung artifact family

Cell nay in phan dau cua report markdown de xem nhanh output co dung y khong.


In [ ]:
for family, run_dir in artifact_dirs.items():
    report_path = run_dir / f'{family}_report.md'
    print('')
    print('=' * 80)
    print(family)
    print(report_path)
    print('-' * 80)
    print(report_path.read_text()[:2000])


## 11. Decision gate closeout

Cell nay tong hop ket luan van hanh sau khi scaffold artifacts da pass assertions.


In [ ]:
closeout = {
    'gate0_run': str(GATE0_RUN),
    'tranche2_scaffold_status': 'scaffold_artifacts_recorded',
    'claim_closed': True,
    'model_training_run': False,
    'efficacy_metrics_computed': False,
    'artifact_dirs': {family: str(path) for family, path in artifact_dirs.items()},
    'next_step': 'review_scaffold_artifacts_then_open_split_registry_lock_and_feature_provenance_population',
    'not_allowed_next': [
        'do_not_train_RIFT_Net_Lite_yet',
        'do_not_run_A3_A4_comparators_yet',
        'do_not_open_efficacy_claim',
    ],
}
print(json.dumps(closeout, indent=2))
